# AlexNet Benchmark

use PY310, py311 not support torch.compile, py39 not support libnvrtc.so compatibility

In [1]:
model_name = "vit-torch"

import torch
from torch import nn
from torchvision import models
#import torch_mlir
import numpy as np
import iree
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll


In [2]:
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])


## Experimental

In [3]:
import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cuda:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model_ = model.to(device)

df = pd.DataFrame()

### PyTorch (Baseline)

In [4]:
from timeit import timeit as ti
def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n
for bs in range(1, 27):
    model = model_
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    print("measuring #", bs)
    baseline_f = timeit("model(image)", 30)
    baseline_b = timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3)
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    #print(baseline_f)
    print(baseline_b)

measuring # 1
21.359183825552464
measuring # 2
17.68622112770875
measuring # 3
23.872872504095238
measuring # 4
30.10884951800108
measuring # 5
38.726636208593845
measuring # 6
44.71619178851446
measuring # 7
51.31936197479566
measuring # 8
58.020009038348995
measuring # 9
68.45760717988014
measuring # 10
71.68946353097756
measuring # 11
77.21530770262082
measuring # 12
80.23984575023253
measuring # 13
92.46546092132728
measuring # 14
98.69392961263657
measuring # 15
105.25592789053917
measuring # 16
111.26734223216772
measuring # 17
123.75919241458178
measuring # 18
135.4158946002523
measuring # 19
136.63322540620962
measuring # 20
137.33507227152586
measuring # 21
156.27709434678158
measuring # 22
157.94456470757723
measuring # 23
171.2488423412045
measuring # 24
188.14640181759992
measuring # 25
187.00014241039753
measuring # 26
194.5601770033439


In [5]:
df.style.hide(axis="index")

In [6]:
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")

ValueError: Could not interpret value `pass` for `x`. An entry with this name does not appear in `data`.

In [ ]:
forward = df[df["pass"] == "Forward"]
forward["acceleration"] = baseline_f / forward["time"]
forward

In [ ]:
backward = df[df["pass"] == "Backward"]
backward["acceleration"] = baseline_b / backward["time"]
backward

In [ ]:
full = df[df["pass"] == "Full"]
full["acceleration"] = (baseline_b + baseline_f) / full["time"]
full

In [ ]:
df = pd.concat([forward, backward, full])
df.to_csv(f"{model_name}.csv")

sns.barplot(df, x="pass", y="acceleration", hue="item")
plt.xlabel("Pass")
plt.ylabel("Acceleration")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-acceleration.png")